# Notebook 3: Confidence Rejection + Koşullu Pipeline + DINOv2 Varyantları

Bu sürüm v5 ana modelini temel alır:
- Ana model: DINOv2 ViT-B/14 + CLS
- Checkpoint: `dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth`
- Output dir: `/content/drive/My Drive/Kirsehir_VPR_DINOv2_v5_Final`

Değişken adlarında eski uyumluluk için `vitb` geçen yerler kalabilir; ana referans artık v5 ViT-B modelidir.

In [ ]:
# ============================================================
# Hücre 1: Kütüphane Kurulumu
# ============================================================
!pip install faiss-cpu folium pytorch-metric-learning scikit-learn -q
!pip install git+https://github.com/cvg/LightGlue.git -q
print("✅ Tüm kütüphaneler kuruldu.")


In [ ]:
# ============================================================
# Hücre 2: Import'lar ve Konfigürasyon
# ============================================================
import os, json, random, zipfile, shutil, math, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
from collections import defaultdict, Counter
from PIL import Image, ImageOps
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from pytorch_metric_learning import losses, miners
from pytorch_metric_learning.samplers import MPerClassSampler
from sklearn.preprocessing import LabelEncoder

import faiss

from lightglue import LightGlue, ALIKED
from lightglue.utils import load_image, rbd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DRIVE_ZIP     = "/content/drive/My Drive/kirsehir_data.zip"
LOCAL_ROOT    = "/content/kirsehir_data"
OUTPUT_DIR    = "/content/drive/My Drive/Kirsehir_VPR_DINOv2_v5_Final"
RESULTS_DIR   = "/content/drive/My Drive/Kirsehir_VPR_Results"

# Eski hücrelerle değişken uyumluluğu için MODEL_NAME_B ana v5 modeli temsil eder.
MODEL_NAME_B  = "dinov2_vitb14"
MODEL_NAME_S  = "dinov2_vits14"
MODEL_POOLING = "cls"
MODEL_CHECKPOINT_NAME = "dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth"
DRIVE_MODEL_PATH = os.path.join(OUTPUT_DIR, MODEL_CHECKPOINT_NAME)

OUTFEATURES   = 512
IMG_SIZE      = 518
BATCH_SIZE    = 128
EPOCHS        = 5
NUM_WORKERS   = 4
TOP_K         = 20
RERANK_TOP_K  = 20
GRID_SIZE     = 0.002
UNFREEZE_LAST_N_BLOCKS = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Cihaz: {DEVICE}")
print(f"✅ Ana model: {MODEL_NAME_B} / {MODEL_POOLING}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Hücre 3: Veri Yükleme ve Bölme (Aynı SEED ile aynı split)
# ============================================================
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(LOCAL_ROOT):
    LOCAL_ZIP = "/content/kirsehir_data.zip"
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
        zf.extractall(LOCAL_ROOT)
    os.remove(LOCAL_ZIP)

entries = os.listdir(LOCAL_ROOT)
if len(entries) == 1 and os.path.isdir(os.path.join(LOCAL_ROOT, entries[0])):
    LOCAL_ROOT = os.path.join(LOCAL_ROOT, entries[0])

def extract_lat_lon_heading(filepath):
    parts = os.path.splitext(os.path.basename(filepath))[0].split("_")
    try:
        return float(parts[0]), float(parts[1]), float(parts[2].replace("h", "") if len(parts)>2 else "0")
    except: return None, None, None

all_images = []
for street_name in sorted(os.listdir(LOCAL_ROOT)):
    street_path = os.path.join(LOCAL_ROOT, street_name)
    if not os.path.isdir(street_path) or street_name == "model": continue
    for fname in sorted(os.listdir(street_path)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")): continue
        fpath = os.path.join(street_path, fname)
        lat, lon, heading = extract_lat_lon_heading(fpath)
        if lat is not None:
            all_images.append({
                "filepath": fpath, "street": street_name,
                "lat": lat, "lon": lon, "heading": heading,
                "point_id": f"{lat:.6f}_{lon:.6f}",
                "block_id": f"{int(lat/GRID_SIZE)}_{int(lon/GRID_SIZE)}"
            })

df_all = pd.DataFrame(all_images).sort_values(by=["point_id", "heading"])
unique_blocks = df_all["block_id"].unique().tolist()
random.shuffle(unique_blocks)
n_blocks = len(unique_blocks)

train_blocks = set(unique_blocks[:int(n_blocks * 0.70)])
test_blocks  = set(unique_blocks[int(n_blocks * 0.70):])

df_train = df_all[df_all["block_id"].isin(train_blocks)].copy()
df_train["class_id"] = df_train["point_id"]
le = LabelEncoder()
df_train["label"] = le.fit_transform(df_train["class_id"])

df_test = df_all[df_all["block_id"].isin(test_blocks)].copy()
point_counts = df_test['point_id'].value_counts()
valid_points = point_counts[point_counts >= 2].index
df_test = df_test[df_test['point_id'].isin(valid_points)]

df_query = df_test.groupby("point_id").tail(1).reset_index(drop=True)
query_filepaths = set(df_query["filepath"])
df_db = df_test[~df_test["filepath"].isin(query_filepaths)].reset_index(drop=True)

print(f"✅ Toplam: {len(df_all):,} | Train: {len(df_train)} | DB: {len(df_db)} | Query: {len(df_query)}")


In [ ]:
# ============================================================
# Hücre 4: Model Tanımları ve Yardımcı Fonksiyonlar
# ============================================================

def open_rgb_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=1).pow(1.0 / self.p)

class VPRDINOv2(nn.Module):
    def __init__(self, backbone_name, out_dim, pooling=MODEL_POOLING,
                 unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS, freeze_all=False):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, verbose=False)
        self.backbone_name = backbone_name
        self.pooling = pooling.lower()

        for p in self.backbone.parameters():
            p.requires_grad = False

        if not freeze_all:
            n_blocks = len(getattr(self.backbone, "blocks", []))
            trainable_blocks = set(range(max(0, n_blocks - unfreeze_last_n), n_blocks))
            for name, param in self.backbone.named_parameters():
                if name.startswith("norm"):
                    param.requires_grad = True
                elif name.startswith("blocks."):
                    parts = name.split(".")
                    if len(parts) > 1 and parts[1].isdigit() and int(parts[1]) in trainable_blocks:
                        param.requires_grad = True

        if self.pooling == "gem":
            self.gem = GeMPooling(p=3)
        elif self.pooling != "cls":
            raise ValueError(f"Desteklenmeyen pooling: {pooling}")

        embed_dim = self.backbone.embed_dim
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.10),
            nn.Linear(embed_dim, out_dim)
        )

    def forward(self, x):
        ret = self.backbone.forward_features(x)
        if self.pooling == "cls":
            x = ret["x_norm_clstoken"]
        else:
            x = self.gem(ret["x_norm_patchtokens"])
        x = self.head(x)
        return nn.functional.normalize(x, p=2, dim=1)


class VPRTrainDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(open_rgb_image(row["filepath"])), int(row["label"])

class StandardVPRDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return self.transform(open_rgb_image(self.df.iloc[idx]["filepath"])), idx


eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.55, 1.0), ratio=(0.75, 1.33),
                        interpolation=T.InterpolationMode.BICUBIC),
    T.RandomApply([T.RandomPerspective(distortion_scale=0.18, p=1.0,
                                        interpolation=T.InterpolationMode.BICUBIC)], p=0.35),
    T.RandomRotation(degrees=7, interpolation=T.InterpolationMode.BICUBIC),
    T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.30, hue=0.04),
    T.RandomAutocontrast(p=0.25),
    T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))], p=0.20),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2, dp, dl = map(math.radians, [lat1, lat2, lat2-lat1, lon2-lon1])
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


@torch.no_grad()
def extract_embeddings(model, df, desc="Emb"):
    loader = DataLoader(StandardVPRDataset(df, eval_transform),
                        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    embs = []
    for imgs, _ in tqdm(loader, desc=desc):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            embs.append(model(imgs).detach().cpu().numpy())
    return np.concatenate(embs, axis=0)


def full_evaluate(query_emb, db_emb, df_query, df_db, top_k=20):
    """Tam değerlendirme: ana metrik Top-1, ek tanı Top-K adaylardır."""
    db_e = db_emb.copy().astype(np.float32)
    qry_e = query_emb.copy().astype(np.float32)
    faiss.normalize_L2(db_e)
    faiss.normalize_L2(qry_e)

    index = faiss.IndexFlatIP(db_e.shape[1])
    index.add(db_e)
    dists, idxs = index.search(qry_e, top_k)

    db_meta = df_db[["filepath","street","lat","lon","heading","point_id"]].to_dict("records")

    errors = []
    correct_street = 0
    street_topk = {5: 0, 10: 0, 20: 0}
    for i in range(len(df_query)):
        q = df_query.iloc[i]
        best = db_meta[int(idxs[i, 0])]
        err = haversine(q["lat"], q["lon"], best["lat"], best["lon"])
        errors.append(err)
        if best["street"] == q["street"]:
            correct_street += 1
        top_streets = [db_meta[int(j)]["street"] for j in idxs[i]]
        for k in street_topk:
            if q["street"] in top_streets[:min(k, len(top_streets))]:
                street_topk[k] += 1

    errors = np.array(errors)
    n = max(1, len(df_query))
    return {
        "errors": errors,
        "dists": dists,
        "idxs": idxs,
        "db_meta": db_meta,
        "street_acc": correct_street / n * 100,
        "street_top5": street_topk[5] / n * 100,
        "street_top10": street_topk[10] / n * 100,
        "street_top20": street_topk[20] / n * 100,
        "median_err": np.median(errors),
        "mean_err": np.mean(errors),
        "r1_25": (errors <= 25).mean() * 100,
        "r1_50": (errors <= 50).mean() * 100,
        "r1_100": (errors <= 100).mean() * 100,
        "r1_500": (errors <= 500).mean() * 100,
    }


def print_results(name, r):
    print(f"\n{'─'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")
    print(f"  Sokak Doğruluğu : {r['street_acc']:.2f}%")
    print(f"  Sokak Top-5/10/20: {r['street_top5']:.2f}% / {r['street_top10']:.2f}% / {r['street_top20']:.2f}%")
    print(f"  Medyan Hata     : {r['median_err']:.1f}m")
    print(f"  Ortalama Hata   : {r['mean_err']:.1f}m")
    print(f"  R@1 <25m        : {r['r1_25']:.2f}%")
    print(f"  R@1 <50m        : {r['r1_50']:.2f}%")
    print(f"  R@1 <100m       : {r['r1_100']:.2f}%")
    print(f"  R@1 <500m       : {r['r1_500']:.2f}%")


print("✅ v5 model tanımları hazır.")

In [ ]:
# ============================================================
# Hücre 5: v5 ViT-B/14 Modelini Yükle ve Değerlendir
# ============================================================
print("⏳ v5 ViT-B/14 (fine-tuned) model yükleniyor...")

if not os.path.exists(DRIVE_MODEL_PATH):
    raise FileNotFoundError(
        f"v5 checkpoint bulunamadı:\n{DRIVE_MODEL_PATH}\n"
        "Önce v5 ana eğitim notebook'unu çalıştır."
    )

# Değişken adı eski hücre uyumluluğu için model_vitb; içerik artık v5 ViT-B.
model_vitb = VPRDINOv2(MODEL_NAME_B, OUTFEATURES, pooling=MODEL_POOLING,
                       unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS).to(DEVICE)
state = torch.load(DRIVE_MODEL_PATH, map_location=DEVICE)
if isinstance(state, dict) and "model_state" in state:
    state = state["model_state"]
model_vitb.load_state_dict(state, strict=True)
model_vitb.eval()

params_b_total = sum(p.numel() for p in model_vitb.parameters())
params_b_train = sum(p.numel() for p in model_vitb.parameters() if p.requires_grad)
print(f"  Toplam param: {params_b_total:,} | Eğitilebilir: {params_b_train:,}")

db_emb_b = extract_embeddings(model_vitb, df_db, "DB (v5 ViT-B)")
qry_emb_b = extract_embeddings(model_vitb, df_query, "Query (v5 ViT-B)")

res_vitb = full_evaluate(qry_emb_b, db_emb_b, df_query, df_db, TOP_K)
print_results("v5 ViT-B/14 Fine-tuned (CLS + MS Loss)", res_vitb)

In [ ]:
# ============================================================
# Hücre 6: CONFIDENCE-BASED REJECTION ANALİZİ
# ============================================================
print(f"\n{'='*65}")
print(f"📊 ANALİZ 1: CONFIDENCE-BASED REJECTION")
print(f"{'='*65}")

# Her sorgu için Top-1 skoru ve hatası
top1_scores = res_vitb["dists"][:, 0]  # cosine similarity skorları
top1_errors = res_vitb["errors"]

# Farklı eşik değerleri ile accuracy-coverage trade-off
thresholds = np.arange(0.60, 0.96, 0.01)

rejection_results = []
for thresh in thresholds:
    # Bu eşiğin üstündeki sorguları kabul et
    accepted_mask = top1_scores >= thresh
    n_accepted = accepted_mask.sum()

    if n_accepted == 0:
        continue

    coverage = n_accepted / len(top1_scores) * 100
    accepted_errors = top1_errors[accepted_mask]

    # Kabul edilen sorgulardaki sokak doğruluğu
    accepted_correct = 0
    db_meta = res_vitb["db_meta"]
    idxs = res_vitb["idxs"]
    for i in range(len(df_query)):
        if not accepted_mask[i]:
            continue
        best = db_meta[idxs[i, 0]]
        if best["street"] == df_query.iloc[i]["street"]:
            accepted_correct += 1

    street_acc = accepted_correct / n_accepted * 100

    rejection_results.append({
        "threshold": thresh,
        "coverage": coverage,
        "n_accepted": n_accepted,
        "n_rejected": len(top1_scores) - n_accepted,
        "street_acc": street_acc,
        "median_err": np.median(accepted_errors),
        "mean_err": np.mean(accepted_errors),
        "r1_25": (accepted_errors <= 25).mean() * 100,
        "r1_50": (accepted_errors <= 50).mean() * 100,
        "r1_100": (accepted_errors <= 100).mean() * 100,
    })

df_rej = pd.DataFrame(rejection_results)

# Önemli eşik noktalarını göster
print(f"\n{'Eşik':>6} {'Kapsam':>8} {'Kabul':>6} {'Red':>5} {'Sokak%':>8} {'Med.Hata':>9} {'<25m':>7} {'<100m':>7}")
print(f"{'─'*65}")
# Tüm veriler (eşik yok)
print(f"{'Yok':>6} {'100.0%':>8} {len(top1_scores):>6} {0:>5} "
      f"{res_vitb['street_acc']:>7.2f}% {res_vitb['median_err']:>8.1f}m "
      f"{res_vitb['r1_25']:>6.2f}% {res_vitb['r1_100']:>6.2f}%")
print(f"{'─'*65}")

for _, row in df_rej[df_rej["threshold"].isin([0.70, 0.75, 0.78, 0.80, 0.82, 0.85, 0.88, 0.90, 0.92])].iterrows():
    print(f"{row['threshold']:>6.2f} {row['coverage']:>7.1f}% {int(row['n_accepted']):>6} {int(row['n_rejected']):>5} "
          f"{row['street_acc']:>7.2f}% {row['median_err']:>8.1f}m "
          f"{row['r1_25']:>6.2f}% {row['r1_100']:>6.2f}%")

# En iyi trade-off noktasını bul (F1-benzeri: accuracy × coverage dengesi)
df_rej["f1_like"] = 2 * (df_rej["r1_100"] * df_rej["coverage"]) / (df_rej["r1_100"] + df_rej["coverage"] + 1e-9)
best_row = df_rej.loc[df_rej["f1_like"].idxmax()]
print(f"\n🎯 Önerilen eşik (accuracy-coverage dengesi): {best_row['threshold']:.2f}")
print(f"   Kapsam: {best_row['coverage']:.1f}% | Sokak Doğ.: {best_row['street_acc']:.1f}% | "
      f"R@1<100m: {best_row['r1_100']:.1f}% | Med. Hata: {best_row['median_err']:.1f}m")


In [ ]:
# ============================================================
# Hücre 7: Confidence Rejection Görselleştirmesi (Makale Figürü)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sol: Accuracy-Coverage Trade-off
ax1 = axes[0]
ax1.plot(df_rej["coverage"], df_rej["r1_100"], 'b-', lw=2, label="R@1 < 100m")
ax1.plot(df_rej["coverage"], df_rej["street_acc"], 'g--', lw=2, label="Street Accuracy")
ax1.plot(df_rej["coverage"], df_rej["r1_25"], 'r:', lw=2, label="R@1 < 25m")
ax1.axvline(x=best_row["coverage"], color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel("Coverage (%)", fontsize=12)
ax1.set_ylabel("Accuracy (%)", fontsize=12)
ax1.set_title("Accuracy–Coverage Trade-off", fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 105)
ax1.set_ylim(0, 105)

# Orta: Medyan hata vs eşik
ax2 = axes[1]
ax2.plot(df_rej["threshold"], df_rej["median_err"], 'b-', lw=2, label="Median Error")
ax2.plot(df_rej["threshold"], df_rej["mean_err"], 'r--', lw=1.5, alpha=0.7, label="Mean Error")
ax2.axvline(x=best_row["threshold"], color='gray', linestyle=':', alpha=0.5,
            label=f"Suggested θ={best_row['threshold']:.2f}")
ax2.set_xlabel("Confidence Threshold (θ)", fontsize=12)
ax2.set_ylabel("Localization Error (m)", fontsize=12)
ax2.set_title("Error vs Confidence Threshold", fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Sağ: Kabul/Red edilen sorguların skor dağılımı (önerilen eşik ile)
optimal_thresh = best_row["threshold"]
accepted = top1_scores >= optimal_thresh
rejected = top1_scores < optimal_thresh

ax3 = axes[2]
ax3.hist(top1_scores[accepted], bins=30, alpha=0.6, color="green",
         label=f"Accepted ({accepted.sum()})", density=True)
ax3.hist(top1_scores[rejected], bins=30, alpha=0.6, color="red",
         label=f"Rejected ({rejected.sum()})", density=True)
ax3.axvline(x=optimal_thresh, color='black', linestyle='--', lw=2,
            label=f"θ = {optimal_thresh:.2f}")
ax3.set_xlabel("DINOv2 Cosine Similarity", fontsize=12)
ax3.set_ylabel("Density", fontsize=12)
ax3.set_title(f"Score Distribution (θ = {optimal_thresh:.2f})", fontsize=13)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "v5_confidence_rejection.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {fig_path}")

# CSV kaydet
df_rej.to_csv(os.path.join(RESULTS_DIR, "v5_confidence_rejection_table.csv"), index=False)
print(f"✅ Tablo kaydedildi.")


In [ ]:
# ============================================================
# Hücre 8: KOŞULLU İKİ AŞAMALI PİPELİNE (Conditional Re-Ranking)
# ============================================================
# Strateji:
#   - v5 DINOv2 Top-1 skoru >= eşik → v5 DINOv2 Top-1'i kabul et (hızlı)
#   - v5 DINOv2 Top-1 skoru <  eşik → LightGlue ile Top-20'yi yeniden sırala (yavaş ama tanısal)

print(f"\n{'='*65}")
print(f"📊 ANALİZ 2: KOŞULLU İKİ AŞAMALI PİPELİNE")
print(f"{'='*65}")

lg_extractor = ALIKED(max_num_keypoints=1024).eval().to(DEVICE)
lg_matcher = LightGlue(features='aliked').eval().to(DEVICE)

# FAISS araması — v5 ViT-B ile Top-20
db_emb_n = db_emb_b.copy().astype(np.float32)
qry_emb_n = qry_emb_b.copy().astype(np.float32)
faiss.normalize_L2(db_emb_n)
faiss.normalize_L2(qry_emb_n)

faiss_index = faiss.IndexFlatIP(OUTFEATURES)
faiss_index.add(db_emb_n)
dists_all, idxs_all = faiss_index.search(qry_emb_n, RERANK_TOP_K)

db_meta = df_db[["filepath","street","lat","lon","heading","point_id"]].to_dict("records")

test_thresholds = [0.70, 0.75, 0.78, 0.80, 0.82, 0.85]

for thresh in test_thresholds:
    needs_lg = (dists_all[:, 0] < thresh).sum()
    print(f"  Eşik {thresh:.2f}: {needs_lg} sorgu LightGlue'ya gidecek "
          f"({needs_lg/len(df_query)*100:.1f}%)")

print(f"\n⚠️ En geniş eşik (0.85) seçilecek, böylece tüm alt kümeler tek seferde hesaplanır.")
print(f"   Daha düşük eşikler bu sonuçların alt kümesi olacak.\n")

In [ ]:
# ============================================================
# Hücre 9: LightGlue — Sadece Düşük Güvenli Sorgular
# ============================================================
# En geniş eşik ile çalıştır, sonra alt eşikler otomatik hesaplanacak

WIDEST_THRESH = 0.85
needs_lightglue = dists_all[:, 0] < WIDEST_THRESH
n_lg = needs_lightglue.sum()

print(f"⚡ {n_lg} sorgu için LightGlue çalıştırılacak (skor < {WIDEST_THRESH})")
print(f"   Kalan {len(df_query) - n_lg} sorgu DINOv2 Top-1 ile kalacak\n")

# LightGlue sonuçlarını sakla (sadece düşük güvenli sorgular için)
lg_rerank_results = {}  # qi -> sorted candidates list
lg_timings = []

lg_indices = np.where(needs_lightglue)[0]

for count, qi in enumerate(tqdm(lg_indices, desc="Koşullu LightGlue")):
    q_row = df_query.iloc[qi]
    t_start = time.time()

    try:
        img0 = load_image(q_row["filepath"]).to(DEVICE)
        feats0 = lg_extractor.extract(img0)
    except Exception as e:
        lg_rerank_results[qi] = None
        lg_timings.append(0)
        continue

    candidates = []
    for k in range(RERANK_TOP_K):
        cand = db_meta[idxs_all[qi, k]]
        try:
            img1 = load_image(cand["filepath"]).to(DEVICE)
            feats1 = lg_extractor.extract(img1)
            matches01 = lg_matcher({"image0": feats0, "image1": feats1})
            matches_clean = rbd(matches01)
            num_matches = len(matches_clean['matches'])

            candidates.append({
                "idx": idxs_all[qi, k],
                "filepath": cand["filepath"],
                "street": cand["street"],
                "lat": cand["lat"], "lon": cand["lon"],
                "num_matches": num_matches,
                "dino_score": float(dists_all[qi, k]),
            })
            del img1, feats1, matches01
        except:
            candidates.append({
                "idx": idxs_all[qi, k],
                "filepath": cand["filepath"],
                "street": cand["street"],
                "lat": cand["lat"], "lon": cand["lon"],
                "num_matches": 0,
                "dino_score": float(dists_all[qi, k]),
            })

    del img0, feats0
    torch.cuda.empty_cache()

    # Match sayısına göre sırala
    candidates.sort(key=lambda x: x["num_matches"], reverse=True)
    lg_rerank_results[qi] = candidates
    lg_timings.append(time.time() - t_start)

    if (count + 1) % 100 == 0:
        avg_t = np.mean(lg_timings[-100:])
        remaining = (n_lg - count - 1) * avg_t / 60
        print(f"   [{count+1}/{n_lg}] {avg_t:.1f}s/sorgu, Kalan: ~{remaining:.0f} dk")

print(f"\n✅ Koşullu LightGlue tamamlandı.")
print(f"   Toplam süre: {sum(lg_timings)/60:.1f} dakika")
print(f"   Ortalama süre/sorgu: {np.mean(lg_timings):.2f}s")


In [ ]:
# ============================================================
# Hücre 10: Koşullu Pipeline Sonuçları — Tüm Eşikler
# ============================================================

test_thresholds = [0.70, 0.75, 0.78, 0.80, 0.82, 0.85]
conditional_results = []

for thresh in test_thresholds:
    errors_cond = []
    correct_street_cond = 0
    n_used_dino = 0
    n_used_lg = 0

    for qi in range(len(df_query)):
        q = df_query.iloc[qi]
        dino_score = dists_all[qi, 0]

        if dino_score >= thresh:
            # DINOv2 Top-1 kabul
            best = db_meta[idxs_all[qi, 0]]
            n_used_dino += 1
        else:
            # LightGlue re-ranking
            if qi in lg_rerank_results and lg_rerank_results[qi] is not None:
                best = lg_rerank_results[qi][0]  # LightGlue'nun 1. sırası
                n_used_lg += 1
            else:
                # LightGlue sonucu yoksa DINOv2 Top-1'e düş
                best = db_meta[idxs_all[qi, 0]]
                n_used_dino += 1

        err = haversine(q["lat"], q["lon"], best["lat"], best["lon"])
        errors_cond.append(err)

        if best["street"] == q["street"]:
            correct_street_cond += 1

    errors_cond = np.array(errors_cond)

    conditional_results.append({
        "threshold": thresh,
        "n_dino": n_used_dino,
        "n_lg": n_used_lg,
        "lg_pct": n_used_lg / len(df_query) * 100,
        "street_acc": correct_street_cond / len(df_query) * 100,
        "median_err": np.median(errors_cond),
        "mean_err": np.mean(errors_cond),
        "r1_25": (errors_cond <= 25).mean() * 100,
        "r1_50": (errors_cond <= 50).mean() * 100,
        "r1_100": (errors_cond <= 100).mean() * 100,
        "r1_500": (errors_cond <= 500).mean() * 100,
        "errors": errors_cond,
    })

df_cond = pd.DataFrame(conditional_results)

print(f"\n{'='*90}")
print(f"📊 KOŞULLU PİPELİNE SONUÇLARI (DINOv2 + Adaptive LightGlue)")
print(f"{'='*90}")
print(f"{'Eşik':>6} {'DINOv2':>7} {'LG':>5} {'LG%':>6} {'Sokak%':>8} {'Med.(m)':>8} "
      f"{'<25m':>7} {'<50m':>7} {'<100m':>7} {'<500m':>7}")
print(f"{'─'*90}")

# Önce DINOv2-only (referans)
print(f"{'–':>6} {len(df_query):>7} {0:>5} {'0.0%':>6} "
      f"{res_vitb['street_acc']:>7.2f}% {res_vitb['median_err']:>7.1f} "
      f"{res_vitb['r1_25']:>6.2f}% {res_vitb['r1_50']:>6.2f}% "
      f"{res_vitb['r1_100']:>6.2f}% {res_vitb['r1_500']:>6.2f}%  ← DINOv2 Only")
print(f"{'─'*90}")

for _, row in df_cond.iterrows():
    marker = ""
    # DINOv2-only'den iyi mi?
    if row["r1_100"] > res_vitb["r1_100"]:
        marker = " ✓"
    print(f"{row['threshold']:>6.2f} {int(row['n_dino']):>7} {int(row['n_lg']):>5} "
          f"{row['lg_pct']:>5.1f}% {row['street_acc']:>7.2f}% {row['median_err']:>7.1f} "
          f"{row['r1_25']:>6.2f}% {row['r1_50']:>6.2f}% "
          f"{row['r1_100']:>6.2f}% {row['r1_500']:>6.2f}%{marker}")

# En iyi koşullu pipeline'ı bul
best_cond = max(conditional_results, key=lambda x: x["r1_100"])
improvement = best_cond["r1_100"] - res_vitb["r1_100"]
print(f"\n🎯 En iyi koşullu eşik: {best_cond['threshold']:.2f}")
print(f"   R@1<100m iyileşme: {res_vitb['r1_100']:.2f}% → {best_cond['r1_100']:.2f}% "
      f"({'+'if improvement>0 else ''}{improvement:.2f}%)")
print(f"   LightGlue kullanım oranı: {best_cond['lg_pct']:.1f}%")


In [ ]:
# ============================================================
# Hücre 11: Koşullu Pipeline Görselleştirmesi (Makale Figürü)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sol: CDF — DINOv2 Only vs En İyi Koşullu Pipeline
best_cond_data = max(conditional_results, key=lambda x: x["r1_100"])

sd_dino = np.sort(res_vitb["errors"])
cdf_dino = np.arange(1, len(sd_dino)+1) / len(sd_dino) * 100

sd_cond = np.sort(best_cond_data["errors"])
cdf_cond = np.arange(1, len(sd_cond)+1) / len(sd_cond) * 100

axes[0].plot(sd_dino, cdf_dino, 'b-', lw=2, label="DINOv2 Only")
axes[0].plot(sd_cond, cdf_cond, 'r-', lw=2,
             label=f"Conditional (θ={best_cond_data['threshold']:.2f})")
axes[0].set_xlim(0, 500)
axes[0].set_xlabel("Localization Error (meters)", fontsize=12)
axes[0].set_ylabel("Percentage of Queries (%)", fontsize=12)
axes[0].set_title("CDF: DINOv2 Only vs Conditional Pipeline", fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=100, color='gray', linestyle=':', alpha=0.5)

# Sağ: Eşik vs R@1<100m bar chart
thresholds_plot = [r["threshold"] for r in conditional_results]
r100_plot = [r["r1_100"] for r in conditional_results]
lg_pct_plot = [r["lg_pct"] for r in conditional_results]

ax2 = axes[1]
bars = ax2.bar(range(len(thresholds_plot)), r100_plot, color='#6366f1',
               edgecolor='white', alpha=0.8)
ax2.axhline(y=res_vitb["r1_100"], color='red', linestyle='--', lw=2,
            label=f"DINOv2 Only ({res_vitb['r1_100']:.1f}%)")
ax2.set_xticks(range(len(thresholds_plot)))
ax2.set_xticklabels([f"θ={t:.2f}\n({p:.0f}% LG)" for t, p in zip(thresholds_plot, lg_pct_plot)],
                     fontsize=9)
ax2.set_ylabel("R@1 < 100m (%)", fontsize=12)
ax2.set_title("Conditional Pipeline: Effect of Threshold", fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# Barların üstüne değer yaz
for bar, val in zip(bars, r100_plot):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "v5_conditional_pipeline.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {fig_path}")


In [ ]:
# ============================================================
# Hücre 12: DINOv2 ViT-S/14 Eğitimi (Küçük Model Ablation)
# ============================================================
print(f"\n{'='*65}")
print(f"📊 ANALİZ 3: DINOv2 ViT-S/14 vs v5 ViT-B/14")
print(f"{'='*65}")

print("⏳ ViT-S/14 CLS modeli oluşturuluyor ve eğitiliyor...")
model_vits = VPRDINOv2(MODEL_NAME_S, OUTFEATURES, pooling=MODEL_POOLING,
                       unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS).to(DEVICE)

params_s_total = sum(p.numel() for p in model_vits.parameters())
params_s_train = sum(p.numel() for p in model_vits.parameters() if p.requires_grad)
print(f"  ViT-S Toplam param: {params_s_total:,} | Eğitilebilir: {params_s_train:,}")
print(f"  ViT-B Toplam param: {params_b_total:,} | Eğitilebilir: {params_b_train:,}")
print(f"  Boyut oranı: ViT-S / ViT-B = {params_s_total/params_b_total*100:.1f}%")

train_dataset = VPRTrainDataset(df_train, train_transform)
sampler = MPerClassSampler(df_train["label"].values, m=4, batch_size=BATCH_SIZE,
                           length_before_new_iter=len(df_train))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

miner_fn = miners.MultiSimilarityMiner(epsilon=0.1)
loss_fn = losses.MultiSimilarityLoss(alpha=2, beta=50, base=0.5)

optimizer = optim.AdamW([
    {'params': [p for p in model_vits.backbone.parameters() if p.requires_grad], 'lr': 5e-6},
    {'params': model_vits.head.parameters(), 'lr': 8e-5}
], weight_decay=1e-4)

print(f"\n⏳ ViT-S/14 Fine-Tuning başlıyor ({EPOCHS} epoch)...")
t_train_start = time.time()

for epoch in range(EPOCHS):
    model_vits.train()
    epoch_loss, valid_batches = 0.0, 0
    for imgs, labels in tqdm(train_loader, desc=f"ViT-S Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        embeddings = model_vits(imgs)
        hard_pairs = miner_fn(embeddings, labels)
        loss = loss_fn(embeddings, labels, hard_pairs)
        if torch.isfinite(loss) and loss.item() > 0:
            loss.backward(); optimizer.step()
            epoch_loss += loss.item(); valid_batches += 1
    avg = epoch_loss / valid_batches if valid_batches > 0 else 0
    print(f"   Epoch {epoch+1} Loss: {avg:.4f}")

t_train_total = time.time() - t_train_start
print(f"\n✅ ViT-S eğitim süresi: {t_train_total/60:.1f} dakika")

vits_path = os.path.join(OUTPUT_DIR, "dinov2_vits14_kirsehir_v5_cls_ablation.pth")
torch.save(model_vits.state_dict(), vits_path)
print(f"✅ ViT-S ağırlıkları kaydedildi: {vits_path}")

In [ ]:
# ============================================================
# Hücre 13: ViT-S/14 Değerlendirme ve v5 ViT-B Karşılaştırma
# ============================================================
model_vits.eval()
torch.cuda.empty_cache()

single_times_s = []
for i in range(min(100, len(df_query))):
    tensor = eval_transform(open_rgb_image(df_query.iloc[i]["filepath"])).unsqueeze(0).to(DEVICE)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time()
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        _ = model_vits(tensor)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    single_times_s.append(time.time() - t0)

single_times_b = []
for i in range(min(100, len(df_query))):
    tensor = eval_transform(open_rgb_image(df_query.iloc[i]["filepath"])).unsqueeze(0).to(DEVICE)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time()
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        _ = model_vitb(tensor)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    single_times_b.append(time.time() - t0)

t0 = time.time()
db_emb_s = extract_embeddings(model_vits, df_db, "DB (ViT-S)")
t_db_s = time.time() - t0

t0 = time.time()
qry_emb_s = extract_embeddings(model_vits, df_query, "Query (ViT-S)")
t_qry_s = time.time() - t0

res_vits = full_evaluate(qry_emb_s, db_emb_s, df_query, df_db, TOP_K)

print_results("ViT-S/14 Fine-tuned (CLS + MS Loss)", res_vits)
print_results("v5 ViT-B/14 Fine-tuned (CLS + MS Loss)", res_vitb)

In [ ]:
# ============================================================
# Hücre 14: ViT-S vs v5 ViT-B Karşılaştırma Tablosu ve Görselleştirme
# ============================================================

print(f"\n{'='*80}")
print(f"📊 DINOv2 VARYANT KARŞILAŞTIRMASI")
print(f"{'='*80}")

model_size_s = os.path.getsize(vits_path) / (1024**2) if os.path.exists(vits_path) else 0
vitl_path = DRIVE_MODEL_PATH
model_size_l = os.path.getsize(vitl_path) / (1024**2) if os.path.exists(vitl_path) else 0

comparison = {
    "Metric": [
        "Backbone",
        "Parameters (total)",
        "Model File Size (MB)",
        "Embedding Dim",
        "Single Image Inference (ms)",
        "Street Accuracy (%)",
        "Street Top-20 Accuracy (%)",
        "Median Error (m)",
        "Mean Error (m)",
        "R@1 < 25m (%)",
        "R@1 < 50m (%)",
        "R@1 < 100m (%)",
        "R@1 < 500m (%)",
    ],
    "ViT-S/14": [
        "ViT-Small (384-dim)",
        f"{params_s_total:,}",
        f"{model_size_s:.1f}",
        OUTFEATURES,
        f"{np.median(single_times_s)*1000:.1f}",
        f"{res_vits['street_acc']:.2f}",
        f"{res_vits['street_top20']:.2f}",
        f"{res_vits['median_err']:.1f}",
        f"{res_vits['mean_err']:.1f}",
        f"{res_vits['r1_25']:.2f}",
        f"{res_vits['r1_50']:.2f}",
        f"{res_vits['r1_100']:.2f}",
        f"{res_vits['r1_500']:.2f}",
    ],
    "v5 ViT-B/14": [
        "ViT-Base (768-dim)",
        f"{params_b_total:,}",
        f"{model_size_l:.1f}",
        OUTFEATURES,
        f"{np.median(single_times_b)*1000:.1f}",
        f"{res_vitb['street_acc']:.2f}",
        f"{res_vitb['street_top20']:.2f}",
        f"{res_vitb['median_err']:.1f}",
        f"{res_vitb['mean_err']:.1f}",
        f"{res_vitb['r1_25']:.2f}",
        f"{res_vitb['r1_50']:.2f}",
        f"{res_vitb['r1_100']:.2f}",
        f"{res_vitb['r1_500']:.2f}",
    ]
}

df_comp = pd.DataFrame(comparison)
print(df_comp.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sd_s = np.sort(res_vits["errors"])
cdf_s = np.arange(1, len(sd_s)+1) / len(sd_s) * 100
sd_l = np.sort(res_vitb["errors"])
cdf_l = np.arange(1, len(sd_l)+1) / len(sd_l) * 100

axes[0].plot(sd_l, cdf_l, 'b-', lw=2, label=f"v5 ViT-B/14 ({params_b_total/1e6:.0f}M params)")
axes[0].plot(sd_s, cdf_s, 'r--', lw=2, label=f"ViT-S/14 ({params_s_total/1e6:.0f}M params)")
axes[0].set_xlim(0, 500)
axes[0].set_xlabel("Localization Error (meters)", fontsize=12)
axes[0].set_ylabel("Percentage of Queries (%)", fontsize=12)
axes[0].set_title("CDF: ViT-S/14 vs v5 ViT-B/14", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

metrics = ["Street\nAcc.", "Top-20\nStreet", "R@1\n<50m", "R@1\n<100m"]
vals_l = [res_vitb["street_acc"], res_vitb["street_top20"], res_vitb["r1_50"], res_vitb["r1_100"]]
vals_s = [res_vits["street_acc"], res_vits["street_top20"], res_vits["r1_50"], res_vits["r1_100"]]

x = np.arange(len(metrics))
w = 0.35
axes[1].bar(x - w/2, vals_l, w, label="v5 ViT-B/14", color='#3b82f6', edgecolor='white')
axes[1].bar(x + w/2, vals_s, w, label="ViT-S/14", color='#f59e0b', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics, fontsize=10)
axes[1].set_ylabel("Accuracy (%)", fontsize=12)
axes[1].set_title("Performance Comparison", fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

axes[2].scatter([params_b_total/1e6], [res_vitb["r1_100"]], s=200, c='#3b82f6',
                zorder=5, edgecolors='black', label="v5 ViT-B/14")
axes[2].scatter([params_s_total/1e6], [res_vits["r1_100"]], s=200, c='#f59e0b',
                zorder=5, edgecolors='black', label="ViT-S/14")
axes[2].annotate(f"v5 ViT-B/14\n{np.median(single_times_b)*1000:.0f}ms",
                 (params_b_total/1e6, res_vitb["r1_100"]),
                 textcoords="offset points", xytext=(15, -15), fontsize=10)
axes[2].annotate(f"ViT-S/14\n{np.median(single_times_s)*1000:.0f}ms",
                 (params_s_total/1e6, res_vits["r1_100"]),
                 textcoords="offset points", xytext=(15, 10), fontsize=10)
axes[2].set_xlabel("Parameters (millions)", fontsize=12)
axes[2].set_ylabel("R@1 < 100m (%)", fontsize=12)
axes[2].set_title("Size-Performance Trade-off", fontsize=13)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "v5_vit_variant_comparison.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {fig_path}")

df_comp.to_csv(os.path.join(RESULTS_DIR, "v5_vit_variant_comparison.csv"), index=False)

In [ ]:
# ============================================================
# Hücre 15: NİHAİ BİRLEŞİK ÖZET
# ============================================================

print(f"\n{'='*90}")
print(f"📊 NOTEBOOK 3 v5 — TÜM SONUÇLARIN ÖZETİ")
print(f"{'='*90}")

print(f"\n── 1. Confidence-Based Rejection ──")
print(f"   Optimal eşik: {best_row['threshold']:.2f}")
print(f"   Bu eşikte: Kapsam={best_row['coverage']:.1f}%, "
      f"R@1<100m={best_row['r1_100']:.1f}%, "
      f"Sokak Doğ.={best_row['street_acc']:.1f}%")
print(f"   Eşik olmadan: R@1<100m={res_vitb['r1_100']:.1f}%, "
      f"Sokak Doğ.={res_vitb['street_acc']:.1f}%")

print(f"\n── 2. Koşullu Pipeline (v5 DINOv2 + Adaptive LightGlue) ──")
best_c = max(conditional_results, key=lambda x: x["r1_100"])
print(f"   En iyi eşik: {best_c['threshold']:.2f}")
print(f"   R@1<100m: {res_vitb['r1_100']:.2f}% → {best_c['r1_100']:.2f}% "
      f"(fark: {best_c['r1_100']-res_vitb['r1_100']:+.2f}%)")
print(f"   LightGlue kullanım oranı: {best_c['lg_pct']:.1f}%")
if best_c['r1_100'] > res_vitb['r1_100']:
    print(f"   ✅ Koşullu pipeline DINOv2-only'den DAHA İYİ")
else:
    print(f"   ⚠️ Koşullu pipeline iyileştirme sağlamadı")

print(f"\n── 3. DINOv2 Varyant Karşılaştırması ──")
print(f"   ViT-S/14: {params_s_total/1e6:.0f}M param, "
      f"R@1<100m={res_vits['r1_100']:.2f}%, "
      f"Medyan={res_vits['median_err']:.1f}m, "
      f"Çıkarım={np.median(single_times_s)*1000:.1f}ms")
print(f"   v5 ViT-B/14: {params_b_total/1e6:.0f}M param, "
      f"R@1<100m={res_vitb['r1_100']:.2f}%, "
      f"Medyan={res_vitb['median_err']:.1f}m, "
      f"Çıkarım={np.median(single_times_b)*1000:.1f}ms")
perf_ratio = res_vits['r1_100'] / max(res_vitb['r1_100'], 1e-9) * 100
size_ratio = params_s_total / params_b_total * 100
print(f"   ViT-S performans oranı: {perf_ratio:.1f}% (boyutun {size_ratio:.1f}%'i ile)")

print(f"\n{'='*90}")
print(f"✅ Notebook 3 v5 tamamlandı. Tüm sonuçlar {RESULTS_DIR} klasöründe.")
print(f"{'='*90}")